In [1]:
from dotenv import load_dotenv
import os

# Go up one level from notebooks/ to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

load_dotenv(os.path.join(project_root, '.env'))

project_id = os.getenv('GCP_PROJECT_ID')
credentials_path = os.path.join(project_root, os.getenv('GOOGLE_APPLICATION_CREDENTIALS'))

print(project_id)
print(credentials_path)
print(os.path.exists(credentials_path))  # should print True

dsai-module-2-project-496708
c:\Users\Admin\Desktop\ntu-sctp-dsai\DSAI-MODULE-2-PROJECT\dsai-module-2-project-496708-6b9c53a35141.json
True


In [2]:
from google.cloud import bigquery
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(credentials_path)
client = bigquery.Client(credentials=credentials, project=project_id)
print("Connected to BigQuery!")

Connected to BigQuery!


In [3]:
dataset_id = f'{project_id}.olist_raw'
dataset = bigquery.Dataset(dataset_id)
dataset.location = 'US'
client.create_dataset(dataset, exists_ok=True)
print("Dataset created!")

Dataset created!


In [4]:
import pandas as pd

files = {
    "orders":               "data/olist_orders_dataset.csv",
    "order_items":          "data/olist_order_items_dataset.csv",
    "customers":            "data/olist_customers_dataset.csv",
    "products":             "data/olist_products_dataset.csv",
    "sellers":              "data/olist_sellers_dataset.csv",
    "payments":             "data/olist_order_payments_dataset.csv",
    "reviews":              "data/olist_order_reviews_dataset.csv",
    "geolocation":          "data/olist_geolocation_dataset.csv",
    "category_translation": "data/product_category_name_translation.csv",
}

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

for table_name, filepath in files.items():
    full_path = os.path.join(project_root, filepath)
    df = pd.read_csv(full_path)
    table_id = f"{project_id}.olist_raw.{table_name}"
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    print(f"✓ {table_name} — {len(df)} rows loaded")

✓ orders — 99441 rows loaded
✓ order_items — 112650 rows loaded
✓ customers — 99441 rows loaded
✓ products — 32951 rows loaded
✓ sellers — 3095 rows loaded
✓ payments — 103886 rows loaded
✓ reviews — 99224 rows loaded
✓ geolocation — 1000163 rows loaded
✓ category_translation — 71 rows loaded


In [5]:
for table_name in files.keys():
    table = client.get_table(f"{project_id}.olist_raw.{table_name}")
    print(f"{table_name}: {table.num_rows} rows, {len(table.schema)} columns")

orders: 99441 rows, 8 columns
order_items: 112650 rows, 7 columns
customers: 99441 rows, 5 columns
products: 32951 rows, 9 columns
sellers: 3095 rows, 4 columns
payments: 103886 rows, 5 columns
reviews: 99224 rows, 7 columns
geolocation: 1000163 rows, 5 columns
category_translation: 71 rows, 2 columns
